In [11]:
import random

import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GAE, BatchNorm
from torch_geometric.utils import dropout_adj, from_scipy_sparse_matrix
from torch_geometric.data import Data
from torch.utils.data import DataLoader, TensorDataset
from torch.nn import BatchNorm1d

from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import NMF
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score, f1_score

import scipy.sparse as sp

import networkx as nx
import matplotlib.pyplot as plt

from tqdm import tqdm


import torch
import torch.nn.functional as F
import numpy as np
import scipy.sparse as sp
import pandas as pd

#from torch_geometric.nn import GCNConv, GAE, BatchNorm1d
from torch_geometric.utils import dropout_adj, from_scipy_sparse_matrix
from torch_geometric.data import Data



In [13]:
# Load the data
data = pd.read_csv("/Users/nirwantandukar/input_data/Database.DREAM4/NORMALIZED_EXP/normalized_weight_matrix_all_GRN_100_01_0.2filter.csv", index_col=0)
#data = pd.read_csv("/Users/nirwantandukar/input_data/DREAM5/normalized_weight_matrix_DREAM5_net1_0.2filter.csv", index_col=0)
print(f"Loaded data with shape: {data.shape}")

# Get the list of algorithms
columns = data.columns
algorithms = sorted(set(col.split('.')[0] for col in columns))
print(f"Algorithms found: {algorithms}")

num_genes = data.shape[0]
gene_names = data.index.tolist()
print(f"Number of genes: {num_genes}")

# Node features (identity matrix)
node_features = np.identity(num_genes)
print(f"Node features shape: {node_features.shape}")

Loaded data with shape: (100, 1700)
Algorithms found: ['GRN_ARACNE', 'GRN_BC3NET', 'GRN_C3NET', 'GRN_CLR', 'GRN_GENIE3_ET', 'GRN_GENIE3_RF', 'GRN_GRENIST', 'GRN_GRNBOOST2', 'GRN_INFERELATOR', 'GRN_KBOOST', 'GRN_LEAP', 'GRN_MEOMI', 'GRN_MRNET', 'GRN_MRNETB', 'GRN_NONLINEARODES', 'GRN_PCIT', 'GRN_TIGRESS']
Number of genes: 100
Node features shape: (100, 100)


In [15]:
# ---------------------------
# Helper: Normalized Laplacian
# ---------------------------
def normalize_adj(adj):
    """Compute the normalized Laplacian of an adjacency matrix."""
    adj = sp.coo_matrix(adj)
    rowsum = np.array(adj.sum(1)).flatten()
    # Add a small constant for numerical stability
    d_inv_sqrt = np.power(rowsum + 1e-5, -0.5)
    d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0.0
    d_mat_inv_sqrt = sp.diags(d_inv_sqrt)
    identity = sp.eye(adj.shape[0])
    laplacian = identity - d_mat_inv_sqrt.dot(adj).dot(d_mat_inv_sqrt)
    return laplacian.tocoo()


# ---------------------------
# GCN Encoder (shared across views)
# ---------------------------
class GCNEncoder(torch.nn.Module):
    ef __init__(self, in_channels, out_channels, dropout_rate=0.1):
        super().__init__()
        self.conv1 = GCNConv(in_channels, 2*out_channels)
        self.conv2 = GCNConv(2*out_channels, out_channels)
        self.dropout = torch.nn.Dropout(p=dropout_rate)
        
    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = self.dropout(x)
        x = self.conv2(x, edge_index)
        return x  # No final activation (decoder handles it)
    #def __init__(self, in_channels, hidden_channels, out_channels, num_layers=4, 
    #             dropout_rate=0.01, edge_dropout_rate=0.01):
    #    super(GCNEncoder, self).__init__()
    #    self.num_layers = num_layers
    #    self.dropout_rate = dropout_rate
    #    self.edge_dropout_rate = edge_dropout_rate
        
    #    # First layer: input to 2x hidden size
    #   self.convs = torch.nn.ModuleList()
    #    self.convs.append(GCNConv(in_channels, 2 * hidden_channels, cached=False))
        
    #    # Add intermediate layers with the same dimensions
    #    for _ in range(num_layers - 2):
    #        self.convs.append(GCNConv(2 * hidden_channels, 2 * hidden_channels, cached=False))
        
    #    # Final layer: hidden size to output embedding size
    #    self.convs.append(GCNConv(2 * hidden_channels, out_channels, cached=False))
    
     #   # Batch normalization layers for each intermediate layer
     #   self.bns = torch.nn.ModuleList([BatchNorm1d(2 * hidden_channels, momentum=0.1) 
                                        for _ in range(num_layers - 1)])
    
      #  self.dropout = torch.nn.Dropout(p=dropout_rate)
       # self.activation_hidden = torch.nn.ReLU  # for hidden layers
        #self.activation_output = torch.nn.Sigmoid()  # for final layer (optional)

    #def forward(self, x, edge_index):
    #    # Apply edge dropout to the connectivity
    #    edge_index, _ = dropout_adj(edge_index, p=self.edge_dropout_rate, training=self.training)
        
    #    # First layer (without batch norm)
    #    x = self.convs[0](x, edge_index)
    #    x = F.relu(x)
    
    #    # Intermediate layers: conv -> batch norm -> relu -> dropout -> residual
     #   for i in range(1, self.num_layers - 1):
     #       x_residual = x
     #       x = self.convs[i](x, edge_index)
     #       x = self.bns[i-1](x)
     ##       x = F.relu(x)
     ##       x = self.dropout(x)
     #       x = x + x_residual  # residual connection
     #       
        # Final layer (no batch norm or dropout)
      #  x = self.convs[-1](x, edge_index)
      #  x = F.relu(x)
      #  return x


class SimpleGCNEncoder(torch.nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, 2*out_channels)
        self.conv2 = GCNConv(2*out_channels, out_channels)
        
    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        return self.conv2(x, edge_index)

class EdgeMLPDecoder(torch.nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.mlp = torch.nn.Sequential(
            torch.nn.Linear(2*emb_dim, 64),
            torch.nn.ReLU(),
            torch.nn.Linear(64, 1)
        
    def forward(self, z, edge_index):
        src, dst = edge_index
        edge_features = torch.cat([z[src], z[dst]], dim=1)
        return torch.sigmoid(self.mlp(edge_features).squeeze()
                             
# ---------------------------
# Joint Multi-View GAE Training (shared encoder + TF soft mask)
# ---------------------------
def apply_joint_gae_to_adjacency(
        data, algorithms, node_features,
        embedding_dim=16, epochs=200,
        dropout_rate=0.1, edge_dropout_rate=0.01,
        weight_decay=1e-8,
        is_tf=None,            # 1-D bool / 0-1 vector length N
        epsilon=0.3):          # weight for non-TF rows (0.2-0.5 typical)

    num_genes = node_features.shape[0]

    # ── 1. preprocess every algorithm view ───────────────────────────
    graph_data = {}
    for alg in algorithms:
        cols = [c for c in data.columns if c.startswith(alg + '.')]
        if len(cols) != num_genes:
            print(f"⚠️  Skipping {alg}: expected {num_genes} cols, got {len(cols)}")
            continue
        mat = data[cols].values.reshape(num_genes, num_genes)

        adj_sparse    = sp.csr_matrix(mat)
        adj_norm      = normalize_adj(adj_sparse + sp.eye(num_genes))
        edge_index, _ = from_scipy_sparse_matrix(adj_norm)

        graph_data[alg] = dict(edge_index=edge_index, raw_adj=mat)

    # ── 2. tensors & model ───────────────────────────────────────────
    x       = torch.tensor(node_features, dtype=torch.float32)
    #encoder = GCNEncoder(in_channels=x.shape[1],
    #                     hidden_channels=embedding_dim,
    #                     out_channels=embedding_dim,
    #                     dropout_rate=dropout_rate,
    #                     edge_dropout_rate=edge_dropout_rate)
    #model   = GAE(encoder)
    # Model setup
    encoder = SimpleGCNEncoder(x.shape[1], embedding_dim)
    decoder = EdgeMLPDecoder(embedding_dim)
    model = torch.nn.ModuleDict({'encoder': encoder, 'decoder': decoder})

    device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    x       = x.to(device)
    model   = model.to(device)

    #optimizer = torch.optim.Adam(model.parameters(), lr=0.01,
     #                            weight_decay=weight_decay)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.1)  # High LR
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                    optimizer, mode='min', patience=10)

    # -----------------------------------------------------------
    #   SOFT PRIOR — outgoing TF→* edges full weight = 1
    #                all other rows weight = ε  (e.g. 0.3)
    # -----------------------------------------------------------
    epsilon = 0.5                              # tune 0.2-0.5 later
    tf_idx  = np.where(is_tf)[0] if is_tf is not None else []
    row_weight = torch.full((num_genes,), epsilon, device=device)
    row_weight[tf_idx] = 1.0                   # TF rows weight = 1

    W     = row_weight[:, None].repeat(1, num_genes)  # broadcast to N×N
    W.fill_diagonal_(0.0)                      # remove self-edges
    W_sum = W.sum() + 1e-8                     # avoid division by zero
    # -----------------------------------------------------------

    
    # ── 3. build the TF-soft weight mask ONCE ────────────────────────
    if is_tf is None:
        row_weight = torch.ones(num_genes, device=device)
    else:
        is_tf_torch = torch.tensor(is_tf, dtype=torch.bool, device=device)
        row_weight  = torch.full((num_genes,), epsilon, device=device)
        row_weight[is_tf_torch] = 1.0                 # TF rows weight = 1

    W     = row_weight[:, None].repeat(1, num_genes)  # broadcast to N×N
    W.fill_diagonal_(0.0)                             # no self-edges
    W_sum = W.sum() + 1e-8                            # avoid /0

    # ── 4. training ─────────────────────────────────────────────────
    model.train()
    for epoch in range(1, epochs + 1):
        optimizer.zero_grad()
        total_loss = 0.0

        for alg, gdata in graph_data.items():
            edge_index  = gdata["edge_index"].to(device)
            z           = model.encode(x, edge_index)
            recon_full  = model.decoder.forward_all(z)     # N×N  (logits already in (0,1))

            # --- binary target (no Laplacian) ---
            #target_full = torch.tensor((gdata["raw_adj"] > 0).astype(float),
             #                          dtype=torch.float32,
              #                         device=device)
            cut = np.percentile(gdata["raw_adj"], 95)
            target_full = (gdata["raw_adj"] > cut).astype(float)

            # --- weighted BCE (outgoing TF rows full weight) ---
            bce   = F.binary_cross_entropy(recon_full, target_full, reduction='none')
            loss  = (bce * W).sum() / W_sum

            total_loss += loss / len(graph_data)   # average loss per algorithm

        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        scheduler.step(total_loss)

        if epoch % 50 == 0:
            print(f"epoch {epoch:3d}   loss {total_loss.item():.4f}")

    # ── 5. reconstruct each view ────────────────────────────────────
    model.eval()
    reconstructed = {}
    with torch.no_grad():
        for alg, gdata in graph_data.items():
            ei = gdata['edge_index'].to(device)
            z  = model.encode(x, ei)
            adj_rec = model.decoder.forward_all(z).cpu().numpy()

            print(f"{alg:15s}  min={adj_rec.min():.4f}  max={adj_rec.max():.4f}")

            # safe [0,1] normalisation
            a_min, a_max = adj_rec.min(), adj_rec.max()
            scale = (a_max - a_min) if (a_max - a_min) > 1e-8 else 1.0
            reconstructed[alg] = (adj_rec - a_min) / scale

    return reconstructed


def apply_joint_gae_to_adjacency(
        data, algorithms, node_features,
        embedding_dim=64, epochs=50,
        edge_dropout_rate=0.3,
        is_tf=None,
        epsilon=0.3):

    num_genes = node_features.shape[0]
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # 1. Preprocess graphs - keep edge weights
    graph_data = {}
    for alg in algorithms:
        cols = [c for c in data.columns if c.startswith(alg + '.')]
        if len(cols) != num_genes:
            print(f"Skipping {alg} - shape mismatch")
            continue
            
        mat = data[cols].values.reshape(num_genes, num_genes)
        adj = sp.csr_matrix(mat)
        
        # Normalize keeping edge weights
        adj_norm = normalize_adj(adj)
        edge_index, edge_weight = from_scipy_sparse_matrix(adj_norm)
        
        graph_data[alg] = {
            'edge_index': edge_index.to(device),
            'edge_weight': edge_weight.to(device).float(),
            'raw_adj': mat
        }

    # 2. Initialize model
    x = torch.tensor(node_features, dtype=torch.float32).to(device)
    encoder = SimpleGCNEncoder(x.shape[1], embedding_dim)
    decoder = EdgeMLPDecoder(embedding_dim)
    model = nn.ModuleDict({'encoder': encoder, 'decoder': decoder}).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.1)

    # 3. TF Weighting - Focus on actual edges
    tf_mask = torch.zeros((num_genes, num_genes), device=device)
    if is_tf is not None:
        is_tf_tensor = torch.tensor(is_tf, device=device).bool()
        for alg in graph_data.values():
            src_nodes = alg['edge_index'][0]
            tf_sources = is_tf_tensor[src_nodes]
            tf_mask[alg['edge_index'][:, tf_sources]] = 1.0

    # 4. Training loop with diagnostics
    for epoch in range(1, epochs+1):
        model.train()
        optimizer.zero_grad()
        total_loss = 0
        
        for alg, gdata in graph_data.items():
            edge_index = gdata['edge_index']
            edge_weight = gdata['edge_weight']
            
            # Encode
            z = model['encoder'](x, edge_index)
            
            # Decode only existing edges
            pred = model['decoder'](z, edge_index)
            target = edge_weight
            
            # Apply TF weights
            weights = torch.where(
                tf_mask[edge_index[0], 
                torch.tensor(1.0, device=device), 
                torch.tensor(epsilon, device=device)
            )
            
            loss = F.binary_cross_entropy(pred, target, weight=weights)
            total_loss += loss
        
        total_loss.backward()
        
        # Gradient monitoring
        avg_grad = sum(p.grad.abs().mean() for p in model.parameters() if p.grad is not None)
        print(f"Epoch {epoch:03d} | Loss: {total_loss.item():.4f} | Grad: {avg_grad:.4f}")
        
        optimizer.step()
        
        # Validation sampling
        if epoch % 10 == 0:
            model.eval()
            with torch.no_grad():
                rand_alg = np.random.choice(list(graph_data.keys()))
                edge_index = graph_data[rand_alg]['edge_index']
                pred = model['decoder'](model['encoder'](x, edge_index), edge_index)
                print(f"Sample pred: {pred.min():.3f}-{pred.max():.3f} (mean {pred.mean():.3f})")
            
            model.train()

    # 5. Final reconstruction
    model.eval()
    reconstructed = {}
    with torch.no_grad():
        for alg, gdata in graph_data.items():
            edge_index = gdata['edge_index']
            z = model['encoder'](x, edge_index)
            
            # Reconstruct all possible edges
            row, col = torch.tril_indices(num_genes, num_genes)
            full_edge_index = torch.stack([row, col], dim=0).to(device)
            adj_rec = model['decoder'](z, full_edge_index)
            adj_rec = adj_rec.reshape(num_genes, num_genes).cpu().numpy()
            
            # Store raw scores without normalization
            reconstructed[alg] = adj_rec
            print(f"{alg:15} Min: {adj_rec.min():.3f} Max: {adj_rec.max():.3f} Mean: {adj_rec.mean():.3f}")

    return reconstructed

# ---------------------------
# Running Joint Multi–View GAE with Filtering
# ---------------------------
def run_joint_gae_with_algo_filtering(data, algorithms, node_features, thresholds, 
                                      num_iterations=2, algo_threshold=2,
                                      embedding_dim=16, epochs=200, dropout_rate=0.1, 
                                      edge_dropout_rate=0.01, weight_decay=1e-8,
                                     is_tf=None, epsilon=0.5):
    """
    Run the joint (multi–view) GAE training repeatedly and filter the results across iterations.
    
    For each iteration, the joint GAE is trained on all algorithms/views simultaneously,
    then for each algorithm the reconstructed matrix is binarized using the threshold.
    Finally, only edges that appear in at least `algo_threshold` iterations are retained.
    
    Parameters:
        data (pd.DataFrame): The input DataFrame (100×1700).
        algorithms (list): List of algorithm prefixes.
        node_features (np.array): Shared node features (100×100).
        thresholds (list): List of binarization thresholds (e.g., [0, 0.4, ...]).
        num_iterations (int): Number of GAE runs.
        algo_threshold (int): Minimum number of iterations an edge must be present.
        embedding_dim, epochs, dropout_rate, edge_dropout_rate, weight_decay: Hyperparameters.
        
    Returns:
        filtered_results (dict): Nested dict: threshold → algorithm → filtered (binary) 100×100 matrix.
    """
    num_genes = data.shape[0]
    filtered_results = {}

    for threshold in thresholds:
        print(f"\n🔹 Running Joint GAE with binarization threshold: {threshold}")
        # Prepare storage for binary reconstructed matrices per algorithm over iterations
        filtered_results[f'threshold_{threshold}'] = {alg: [] for alg in algorithms}

        for iteration in range(num_iterations):
            print(f"  ⏳ Iteration {iteration + 1}/{num_iterations}")
            reconstructed_dict = apply_joint_gae_to_adjacency(
                data=data,
                algorithms=algorithms,
                node_features=node_features,
                embedding_dim=embedding_dim,
                epochs=epochs,
                dropout_rate=dropout_rate,
                edge_dropout_rate=edge_dropout_rate,
                weight_decay=weight_decay,
                is_tf=is_tf,           # ← add
                epsilon=epsilon)
            for alg in algorithms:
                rec = reconstructed_dict.get(alg)
                if rec is not None:
                    # Binarize the reconstructed matrix using the threshold
                    binary = (rec > threshold).astype(int)
                    filtered_results[f'threshold_{threshold}'][alg].append(binary)

        # Filter edges across iterations: keep only edges seen in ≥ algo_threshold iterations
        for alg in algorithms:
            matrices = np.stack(filtered_results[f'threshold_{threshold}'][alg], axis=0)
            filtered = (np.sum(matrices, axis=0) >= algo_threshold).astype(int)
            filtered_results[f'threshold_{threshold}'][alg] = filtered

    return filtered_results



SyntaxError: unmatched ']' (3677772205.py, line 59)

In [ ]:
# ---------------------------
# Example usage:
# ---------------------------
# Define parameters for filtering
thresholds = [0.2]         # Example threshold(s)
num_iterations = 5      # Number of runs per threshold
algo_threshold = 2       # Minimum iterations an edge must appear in

tf_list = ('G5', 'G37', 'G26')
# boolean vector length N
is_tf_vec = np.isin(gene_names, tf_list)

# Run joint multi–view GAE with filtering
filtered_results = run_joint_gae_with_algo_filtering(
    data=data,
    algorithms=algorithms,
    node_features=node_features,
    thresholds=thresholds,
    num_iterations=num_iterations,
    algo_threshold=algo_threshold,
    embedding_dim=32,
    epochs=500,
    dropout_rate=0.01,
    edge_dropout_rate=0.3,
    weight_decay=0.0,
    is_tf=is_tf_vec,
    epsilon=0.3
)


In [9]:
filtered_results

NameError: name 'filtered_results' is not defined

In [82]:
# Applying NMF to 17 algorithm
def fuse_with_joint_nmf_from_filtered(filtered_results, threshold, rank=20, num_genes=100):
    """
    Fuses multiple filtered binary adjacency matrices using joint NMF.
    Each matrix must be (num_genes x num_genes), one per algorithm.
    """
    all_flattened = []

    for alg, matrix in filtered_results[f'threshold_{threshold}'].items():
        assert matrix.shape == (num_genes, num_genes), f"{alg} shape is invalid: {matrix.shape}"
        flattened = matrix.flatten()
        all_flattened.append(flattened)

    # all_flattened = 17 x 10000 → each row is one algorithm's flattened adjacency matrix
    nmf_input = np.array(all_flattened)  # shape: (17, 10000)
    print(f"Matrix shape (before transpose): {nmf_input.shape}")

    V = nmf_input.T  # (10000, 17)
    print(f"NMF input shape: {V.shape}")

    nmf_model = NMF(n_components=rank, init='random', random_state=42, max_iter=1000)
    W = nmf_model.fit_transform(V)
    H = nmf_model.components_

    # Reconstruct the fused graph
    V_reconstructed = np.dot(W, H).mean(axis=1)  # shape: (10000,)

    # RESHAPE using known num_genes = 100, not inferred!
    fused_adj = V_reconstructed.reshape((num_genes, num_genes))

    # Normalize to [0, 1]
    fused_adj = (fused_adj - fused_adj.min()) / (fused_adj.max() - fused_adj.min())

    return fused_adj



In [84]:
# Running the NMF fusion
fused_adj_nmf = fuse_with_joint_nmf_from_filtered(filtered_results, threshold=0.2, rank=20)
threshold = 0.2
fused_binary = (fused_adj_nmf >= threshold).astype(int)

Matrix shape (before transpose): (17, 10000)
NMF input shape: (10000, 17)


In [86]:
# Function to extract nodes and edges from the binary matrix
def count_nodes_edges(final_matrix):
    # Count edges in the upper triangle (to avoid double counting for undirected graphs)
    num_edges = np.sum(np.triu(final_matrix, k=1))
    
    # Count nodes (genes with at least one edge, i.e., rows/columns with non-zero values)
    connected_nodes = np.sum(np.any(final_matrix, axis=1))  # Count rows with non-zero values
    num_nodes = connected_nodes
    
    return num_nodes, num_edges

# Function to extract gene interactions (edges) from a binarized matrix
def extract_gene_interactions(final_matrix, gene_names):
    edges = []
    num_genes = final_matrix.shape[0]

    # Loop through the upper triangle of the matrix to avoid duplicates (since it's undirected)
    for i in range(num_genes):
        for j in range(i + 1, num_genes):
            if final_matrix[i, j] == 1:  # Check if there's an interaction
                edges.append(f"{gene_names[i]}-{gene_names[j]}")
    
    return edges


In [88]:


# Function to extract nodes and edges from the binary matrix
def count_nodes_edges(final_matrix):
    # Count edges in the upper triangle (to avoid double counting for undirected graphs)
    num_edges = np.sum(np.triu(final_matrix, k=1))
    
    # Count nodes (genes with at least one edge, i.e., rows/columns with non-zero values)
    connected_nodes = np.sum(np.any(final_matrix, axis=1))  # Count rows with non-zero values
    num_nodes = connected_nodes
    
    return num_nodes, num_edges

# Function to extract gene interactions (edges) from a binarized matrix
def extract_gene_interactions(final_matrix, gene_names):
    edges = []
    num_genes = final_matrix.shape[0]

    # Loop through the upper triangle of the matrix to avoid duplicates (since it's undirected)
    for i in range(num_genes):
        for j in range(i + 1, num_genes):
            if final_matrix[i, j] == 1:  # Check if there's an interaction
                edges.append(f"{gene_names[i]}-{gene_names[j]}")
    
    return edges


In [90]:
# Count nodes and edges
num_nodes, num_edges = count_nodes_edges(fused_binary)
print(f"Nodes: {num_nodes}, Edges: {num_edges}")

# Get the edge list
nmf_edge_list = extract_gene_interactions(fused_binary, gene_names)
print(f"Sample edges: {nmf_edge_list[:5]}")
nmf_edge_list

Nodes: 1, Edges: 0
Sample edges: []


[]

In [68]:
# Convert ground truth matrix to edge list
def convert_matrix_to_edge_list(matrix):
    edges = []
    for source in matrix.index:
        for target in matrix.columns:
            if matrix.at[source, target] == 1:
                edges.append((source, target))
    return edges

# Load known (ground truth) interactions
file_path = "/Users/nirwantandukar/input_data/Database.DREAM4/GS/dream4_100_01_gs.csv"
ground_truth = pd.read_csv(file_path, index_col=0)
print(f"Ground truth loaded with shape: {ground_truth.shape}")

# Get the ground truth edges
known_edges_set = set(convert_matrix_to_edge_list(ground_truth))
print(f"Number of known edges: {len(known_edges_set)}")

def convert_str_edges_to_tuples(edge_list):
    edge_tuples = set()
    for edge in edge_list:
        a, b = edge.split('-')
        # Add both (a, b) and (b, a) to ensure directionality doesn't affect matching
        edge_tuples.add((a, b))
        edge_tuples.add((b, a))
    return edge_tuples

predicted_edge_set = convert_str_edges_to_tuples(nmf_edge_list)

def symmetrize_edges(edge_set):
    symmetric_set = set()
    for a, b in edge_set:
        symmetric_set.add((a, b))
        symmetric_set.add((b, a))
    return symmetric_set

known_edges_symmetric = symmetrize_edges(known_edges_set)



Ground truth loaded with shape: (100, 100)
Number of known edges: 175


In [70]:
# True Positives (correct predictions)
true_positives = predicted_edge_set.intersection(known_edges_symmetric)

# False Positives (predicted but not known)
false_positives = predicted_edge_set - known_edges_symmetric

# False Negatives (missed known edges)
false_negatives = known_edges_symmetric - predicted_edge_set

# Precision, Recall, F1
tp = len(true_positives) // 2  # divide by 2 because of symmetric edges
fp = len(false_positives) // 2
fn = len(false_negatives) // 2

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print(f"TP: {tp}, FP: {fp}, FN: {fn}")
print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1 Score: {f1:.4f}")


TP: 0, FP: 0, FN: 168
Precision: 0.0000, Recall: 0.0000, F1 Score: 0.0000


In [24]:
#### AUPR and AUROC

from sklearn.metrics import roc_auc_score, average_precision_score

def compute_auroc_aupr(predicted_adj, ground_truth_df, gene_names):
    """
    Computes AUROC and AUPR given predicted adjacency (scores) and ground truth (binary).
    Assumes both are 100x100 matrices with matching gene order.
    """

    # Convert ground truth DataFrame to a numpy binary matrix
    gt_matrix = ground_truth_df.loc[gene_names, gene_names].values.astype(int)

    # Flatten only upper triangle (excluding diagonal)
    triu_indices = np.triu_indices_from(gt_matrix, k=1)
    gt_flat = gt_matrix[triu_indices]
    pred_flat = predicted_adj[triu_indices]

    # AUROC and AUPR
    auroc = roc_auc_score(gt_flat, pred_flat)
    aupr = average_precision_score(gt_flat, pred_flat)

    print(f"AUROC: {auroc:.4f}")
    print(f"AUPR : {aupr:.4f}")
    
    return auroc, aupr


In [26]:
# Call the scorer
compute_auroc_aupr(fused_adj_nmf, ground_truth, gene_names)

AUROC: 0.5000
AUPR : 0.0184


(0.5, 0.018383838383838384)

In [131]:
filtered_results

{'threshold_0': {'GRN_ARACNE': array([[0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         ...,
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0]]),
  'GRN_BC3NET': array([[0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         ...,
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0]]),
  'GRN_C3NET': array([[0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         ...,
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0]]),
  'GRN_CLR': array([[0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         ...,
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0]]),
  'GRN_GENIE3_ET': array([[0, 0, 0, ..., 0, 0, 0],
  